In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install -q datasets transformers seqeval accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import os
import numpy as np
import torch
from datasets import load_from_disk
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/indic-ner"
DRIVE_PATH = f"{DRIVE_DIR}/naamapadam_hi_100k"

if not os.path.exists(DRIVE_PATH):
    raise FileNotFoundError("Run Phase 1 notebook first and back up to Drive.")

processed = load_from_disk(DRIVE_PATH)
print(processed)

label_names = processed["train"].features["ner_tags"].feature.names
num_labels = len(label_names)
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}
print(id2label)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Mounted at /content/drive
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 13460
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 867
    })
})
{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}
Device: cuda


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    )
    aligned = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labels, prev = [], None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev:
                labels.append(tags[wid])
            else:
                labels.append(-100)
            prev = wid
        aligned.append(labels)
    enc["labels"] = aligned
    return enc

tokenized = processed.map(
    tokenize_and_align,
    batched=True,
    remove_columns=processed["train"].column_names,
    desc="Tokenizing",
)
print(tokenized)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 7.75MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizing:   0%|          | 0/100000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/13460 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/867 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13460
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 867
    })
})


In [ ]:
ex = tokenized["train"][0]
for tid, lab in zip(ex["input_ids"], ex["labels"]):
    print(f"{tokenizer.convert_ids_to_tokens([tid])[0]:>15s}  {id2label.get(lab, '-')}")

          [CLS]  -
          जुलाई  O
           2012  O
             के  O
            अंत  O
            में  O
              ,  O
          घोषित  O
           किया  O
            गया  O
             कि  O
             गॉ  B-PER
          ##ल्ड  -
          ##िंग  -
             का  O
          दूसरा  O
          एल्बम  O
             है  O
           ##लस  -
           ##ीन  -
           नामक  O
             है  O
             और  O
            इसे  O
              8  O
        अक्टूबर  O
           2012  O
             को  O
         रिलीज़  O
           किया  O
          जाएगा  O
              ।  O
          [SEP]  -


In [ ]:
from seqeval.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    y_true, y_pred = [], []
    for pred, lab in zip(preds, labels):
        keep = lab != -100
        y_true.append([id2label[l] for l in lab[keep]])
        y_pred.append([id2label[p] for p in pred[keep]])
    return {
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="/content/indicbert_hi_runs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=200,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	ca

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.186442,0.181186,0.757331,0.794223,0.775338
2,0.160100,0.175734,0.771934,0.793274,0.782459
3,0.144151,0.177921,0.767950,0.799882,0.783591


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18750, training_loss=0.17867816904703776, metrics={'train_runtime': 2227.192, 'train_samples_per_second': 134.699, 'train_steps_per_second': 8.419, 'total_flos': 9252822090777408.0, 'train_loss': 0.17867816904703776, 'epoch': 3.0})

In [ ]:
from seqeval.metrics import classification_report

preds_out = trainer.predict(tokenized["validation"])
preds = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids

y_true, y_pred = [], []
for pred, lab in zip(preds, labels):
    keep = lab != -100
    y_true.append([id2label[l] for l in lab[keep]])
    y_pred.append([id2label[p] for p in pred[keep]])

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

         LOC       0.81      0.86      0.83     10213
         ORG       0.68      0.69      0.68      9786
         PER       0.81      0.85      0.83     10568

   micro avg       0.77      0.80      0.78     30567
   macro avg       0.77      0.80      0.78     30567
weighted avg       0.77      0.80      0.78     30567



In [ ]:
from seqeval.metrics import f1_score

indicbert_f1 = f1_score(y_true, y_pred)
bilstm_f1 = 0.7406

print(f"BiLSTM-CRF   val F1: {bilstm_f1:.4f}")
print(f"IndicBERT    val F1: {indicbert_f1:.4f}")
print(f"Delta:              {indicbert_f1 - bilstm_f1:+.4f}")

BiLSTM-CRF   val F1: 0.7406
IndicBERT    val F1: 0.7836
Delta:              +0.0430


In [ ]:
SAVE_DIR = f"{DRIVE_DIR}/indicbert_hi_best"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to Drive:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to Drive: /content/drive/MyDrive/indic-ner/indicbert_hi_best
